# Разработка модуля геопланирования и кластеризации точек ежедневного посещения

Блокнот демонстрирует подход к распределению точек между менеджерами на основе географической кластеризации и требований по количеству визитов. В качестве входа используется таблица с координатами точек и плановым числом посещений, а на выходе формируется календарь маршрутов по дням.


## Цель и логика исследования

В блокноте решается задача геопланирования: нужно распределить торговые точки между менеджерами и построить календарь посещений на месяц. Для каждой точки известны координаты (`lat`, `lon`) и требуемое количество посещений (`n_visits`).

В работе сравниваются три подхода:

1. классический KMeans без учёта веса точки;
2. KMeans с весами по количеству требуемых посещений;
3. KMeans с весами и дополнительной балансировкой нагрузки между менеджерами.

Основной результат блокнота — третий алгоритм, так как он одновременно учитывает географическую близость, частоту посещений и равномерность нагрузки.


In [ ]:
import pandas as pd
import numpy as np
import requests
import folium
from folium import Element
import time
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from math import radians, sin, cos, sqrt, atan2
import matplotlib.colors as mcolors
from branca.element import Element
from scipy.spatial import ConvexHull

random_state = 42
n_clusters = 3
max_visits_per_day = 12
n_working_days = 22

## Вспомогательные функции расстояний

Ниже задаются функции для оценки расстояний между точками. В блокноте используются два типа расстояния:

- расстояние по дорогам через OSRM — более близко к реальному автомобильному пробегу, но зависит от внешнего сервиса;
- расстояние по координатам через формулу haversine — локальная оценка расстояния по прямой с учётом кривизны Земли.

Эти функции нужны для расчёта дневного пробега и для упорядочивания точек внутри маршрута менеджера.


In [ ]:
# функция оперделения расстояния по дорогам
def road_distance_osrm_km(lat1, lon1, lat2, lon2):
    url = (
        "https://router.project-osrm.org/route/v1/driving/"
        f"{lon1},{lat1};{lon2},{lat2}"
        "?overview=false"
    )

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    time.sleep(0.1)
    data = response.json()

    if data["code"] != "Ok":
        return None

    distance_m = data["routes"][0]["distance"]

    return distance_m / 1000

In [ ]:
#автомобильная линия маршрута по дорогам
def get_osrm_route_geometry(day_df):
    coords = [
        f"{row['lon']},{row['lat']}"
        for _, row in day_df.iterrows()
    ]

    key = tuple(coords)

    if key in route_cache:
        return route_cache[key]

    if len(coords) < 2:
        return []

    url = (
        "https://router.project-osrm.org/route/v1/driving/"
        + ";".join(coords)
        + "?overview=full&geometries=geojson"
    )

    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()

        result = response.json()

        if result.get("code") != "Ok":
            return []

        geometry = result["routes"][0]["geometry"]["coordinates"]

        # OSRM возвращает [lon, lat], Folium ждёт [lat, lon]
        route_points = [
            [lat, lon]
            for lon, lat in geometry
        ]

        route_cache[key] = route_points

        time.sleep(0.2)

        return route_points

    except Exception as e:
        print("OSRM error:", e)
        return []

In [ ]:
# функция оперделения расстояния по координатам
def haversine_km(lat1, lon1, lat2, lon2):

    R = 6371  # радиус Земли в км

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

## Загрузка и исследование данных

## Входные данные

Исходный файл `data.csv` содержит список точек для посещения. Минимально важные поля:

- `point_id` — уникальный идентификатор точки;
- `manager` — исходное назначение менеджера, используется для анализа первоначальной структуры данных;
- `lat`, `lon` — координаты точки;
- `n_visits` — сколько раз точку нужно посетить за период планирования.

Перед построением маршрутов данные проверяются: удаляются строки с пропусками, некорректными координатами и требованиями по визитам, которые невозможно выполнить в рамках заданного числа рабочих дней.


In [ ]:
data = pd.read_csv('data.csv')
data.head()

In [ ]:
print(data.info())
print('\n-------------\nВсего строк:', len(data))
print('Уникальных point_id:', data['point_id'].nunique())
print('Дубликатов:', data['point_id'].duplicated().sum())

In [ ]:
# убираем отсутствующие или поврежденные строки по ТЗ
data = data.dropna()
data = data.loc[data['n_visits'] <= n_working_days]
data = data[
    data['lat'].notna() &
    data['lon'].notna() &
    np.isfinite(data['lat']) &
    np.isfinite(data['lon']) &
    data['lat'].between(-90, 90) &
    data['lon'].between(-180, 180)
].reset_index(drop=True)

In [ ]:
data.pivot_table(index='point_id', columns='manager', values='n_visits', aggfunc='sum', fill_value=0, margins=True)

In [ ]:
# визуализация исходных данных
colors = list(mcolors.TABLEAU_COLORS.values())

this_map = folium.Map(
    prefer_canvas=True,
    attr=' '
)

def plotDot(point):
    color = colors[int(point.manager) % len(colors)]

    folium.CircleMarker(
        location=[point.lat, point.lon],
        radius=2,
        color=color,
        fill=True,
        fill_color=color,
        weight=2
    ).add_to(this_map)


data.apply(plotDot, axis=1)

this_map.fit_bounds(this_map.get_bounds())

this_map.get_root().html.add_child(Element("""
<style>
.leaflet-control-attribution {
    display: none !important;
}
</style>
"""))

this_map

# Вариант 1. Классический KMeans + случайный выбор точек внутри кластера


Первый подход делит точки на географические кластеры только по координатам. После этого внутри каждого кластера точки выбираются случайно на каждый день.

Такой вариант прост и хорошо показывает базовую географическую группировку, но у него есть важное ограничение: он не учитывает, что некоторые точки требуют больше посещений, чем другие. Из-за этого нагрузка менеджеров может получиться неравномерной.


In [ ]:
data['cluster'] = KMeans(n_clusters=n_clusters, random_state=random_state).fit_predict(data[['lat', 'lon']])
data.groupby('cluster').size()

In [ ]:
colors = list(mcolors.TABLEAU_COLORS.values())

this_map = folium.Map(
    prefer_canvas=True,
    attr=' '  # пустая строка не сработает как надо у некоторых версий, поэтому пробел
)

def plotDot(point):
    '''input: series that contains a numeric named latitude and a numeric named longitude
    this function creates a CircleMarker and adds it to your this_map'''
    color = colors[int(point.cluster) % len(colors)]
    folium.CircleMarker(location=[point.lat, point.lon],
                        radius=2, color=color,
                        weight=5).add_to(this_map)

#use df.apply(,axis=1) to "iterate" through every row in your dataframe
data.apply(plotDot, axis = 1)


#Set the zoom to the maximum possible
this_map.fit_bounds(this_map.get_bounds())



this_map.get_root().html.add_child(Element("""
<style>
.leaflet-control-attribution {
    display: none !important;
}
</style>
"""))


this_map

In [ ]:
n_managers = len(data['manager'].unique())
max_capacity = n_managers * max_visits_per_day * n_working_days

### Симуляция посещений для алгоритма 1

На этом шаге моделируется рабочий месяц. Для каждого дня и каждого менеджера выбирается не больше `max_visits_per_day` точек из его кластера. После посещения остаток требуемых визитов по точке уменьшается.

В результате формируется таблица `selected_points_df`, где каждая строка — конкретный визит менеджера в конкретный день.


In [ ]:
# симуляция первого дня
day = 1
data_original = data.copy()

data['row_id'] = data.index
data['visits_remain'] = data['n_visits']
selected_points_df = pd.DataFrame()  # пустой, будет накапливать историю визитов

shuffled = data.sample(frac=1, random_state=42).reset_index(drop=True)

selected_points = {}
for n in range(n_managers):
    cluster_data = shuffled[shuffled['cluster'] == n]
    df_m = cluster_data.iloc[:max_visits_per_day].copy()
    df_m['manager'] = n
    df_m['day'] = day
    df_m['order_in_day'] = range(1, len(df_m) + 1)
    selected_points[n] = df_m.reset_index(drop=True)


new_batch = pd.concat(selected_points.values(), ignore_index=True)


new_batch = new_batch.sort_values(
    ['manager', 'day', 'order_in_day']
).reset_index(drop=True)

new_batch['prev_lat'] = (
    new_batch
    .groupby(['manager', 'day'])['lat']
    .shift()
)

new_batch['prev_lon'] = (
    new_batch
    .groupby(['manager', 'day'])['lon']
    .shift()
)

new_batch['distance_from_prev_km'] = new_batch.apply(
    lambda row: 0 if pd.isna(row['prev_lat']) else haversine_km(
        row['prev_lat'],
        row['prev_lon'],
        row['lat'],
        row['lon']
    ),
    axis=1
)

new_batch['road_distance_from_prev_km'] = new_batch.apply(
    lambda row: 0 if pd.isna(row['prev_lat']) else road_distance_osrm_km(
        row['prev_lat'],
        row['prev_lon'],
        row['lat'],
        row['lon']
    ),
    axis=1
)

daily_distance_road = (
    new_batch
    .groupby(['manager', 'day'])['road_distance_from_prev_km']
    .sum()
    .reset_index(name='daily_distance_road_km')
)

daily_distance_map = (
    new_batch
    .groupby(['manager', 'day'])['distance_from_prev_km']
    .sum()
    .reset_index(name='daily_distance_map_km')
)

new_batch = new_batch.merge(
    daily_distance_road, 
    on=['manager', 'day'],
    how='left'
)

new_batch = new_batch.merge(
    daily_distance_map, 
    on=['manager', 'day'],
    how='left'
)



if 'selected_points_df' in globals() and len(selected_points_df) > 0:
    selected_points_df = pd.concat([selected_points_df, new_batch], ignore_index=True)
else:
    selected_points_df = new_batch


visited_idx = new_batch['point_id']  # или другой id, который ссылается на data
data.loc[data['point_id'].isin(visited_idx), 'visits_remain'] -= 1

# удаляем полностью отработанные точки
data = data[data['visits_remain'] > 0].reset_index(drop=True)

In [ ]:
#проверка работы кода
data.info()

In [ ]:
#проверка работы кода
selected_points_df.info()

In [ ]:
#проверка работы кода
selected_points_df.sample(36)

In [ ]:
selected_points = selected_points_df['point_id'].to_list()
print(len(selected_points))

In [ ]:
data.loc[data['point_id'].isin(selected_points)]

In [ ]:
# симуляция следующих дней
for day in range(2, n_working_days + 1):
    
    shuffled = data.sample(frac=1, random_state=42).reset_index(drop=True)

    selected_points = {}
    for n in range(n_managers):
        cluster_data = shuffled[shuffled['cluster'] == n]
        df_m = cluster_data.iloc[:max_visits_per_day].copy()
        df_m['manager'] = n
        df_m['day'] = day
        df_m['order_in_day'] = range(1, len(df_m) + 1)
        selected_points[n] = df_m.reset_index(drop=True)


    new_batch = pd.concat(selected_points.values(), ignore_index=True)


    new_batch = new_batch.sort_values(
        ['manager', 'day', 'order_in_day']
    ).reset_index(drop=True)

    new_batch['prev_lat'] = (
        new_batch
        .groupby(['manager', 'day'])['lat']
        .shift()
    )

    new_batch['prev_lon'] = (
        new_batch
        .groupby(['manager', 'day'])['lon']
        .shift()
    )

    new_batch['distance_from_prev_km'] = new_batch.apply(
        lambda row: 0 if pd.isna(row['prev_lat']) else haversine_km(
            row['prev_lat'],
            row['prev_lon'],
            row['lat'],
            row['lon']
        ),
        axis=1
    )
    new_batch['road_distance_from_prev_km'] = new_batch.apply(
    lambda row: 0 if pd.isna(row['prev_lat']) else road_distance_osrm_km(
        row['prev_lat'],
        row['prev_lon'],
        row['lat'],
        row['lon']
    ),
    axis=1
    )

    daily_distance_road = (
        new_batch
        .groupby(['manager', 'day'])['road_distance_from_prev_km']
        .sum()
        .reset_index(name='daily_distance_road_km')
    )

    daily_distance_map = (
        new_batch
        .groupby(['manager', 'day'])['distance_from_prev_km']
        .sum()
        .reset_index(name='daily_distance_map_km')
    )

    new_batch = new_batch.merge(
        daily_distance_road, 
        on=['manager', 'day'],
        how='left'
    )

    new_batch = new_batch.merge(
        daily_distance_map, 
        on=['manager', 'day'],
        how='left'
    )


    if 'selected_points_df' in globals() and len(selected_points_df) > 0:
        selected_points_df = pd.concat([selected_points_df, new_batch], ignore_index=True)
    else:
        selected_points_df = new_batch


    visited_idx = new_batch['point_id']  # или другой id, который ссылается на data
    data.loc[data['point_id'].isin(visited_idx), 'visits_remain'] -= 1

    # удаляем полностью отработанные точки
    data = data[data['visits_remain'] > 0].reset_index(drop=True)

In [ ]:
selected_points_df.info()

In [ ]:
selected_points_df['point_id'].nunique()

In [ ]:
print(selected_points_df.groupby('cluster').size())

In [ ]:
selected_points_df.groupby('manager').size()

### Оценка результата алгоритма 1

После симуляции рассчитываются статусы точек: не посещалась, посещалась частично или выполнена полностью. Также строится карта, которая показывает распределение точек и позволяет визуально оценить, насколько удобными получились зоны.


In [ ]:
# визуализация результата

actual = (
    selected_points_df
    .groupby('point_id')
    .size()
    .rename('visits_done')
)

map_data = (
    data_original
    .copy()
    .merge(actual, on='point_id', how='left')
)

map_data['visits_done'] = map_data['visits_done'].fillna(0).astype(int)
map_data['visits_remain'] = map_data['n_visits'] - map_data['visits_done']

map_data['status'] = np.select(
    [
        map_data['visits_done'] == 0,
        map_data['visits_remain'] <= 0
    ],
    [
        'not_started',
        'done'
    ],
    default='partial'
)

point_colors = {
    'done': 'green',
    'partial': 'orange',
    'not_started': 'red'
}

cluster_colors = {
    0: 'blue',
    1: 'purple',
    2: 'darkred'
}

this_map = folium.Map(
    prefer_canvas=True,
    attr=' '
)

def plotDot(point):
    color = point_colors[point.status]

    folium.CircleMarker(
        location=[point.lat, point.lon],
        radius=4,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        weight=2,
        popup=(
            f"point_id: {point.point_id}<br>"
            f"status: {point.status}<br>"
            f"n_visits: {point.n_visits}<br>"
            f"visits_done: {point.visits_done}<br>"
            f"visits_remain: {point.visits_remain}<br>"
            f"cluster: {point.cluster}"
        )
    ).add_to(this_map)


for cluster_id, cluster_df in map_data.groupby('cluster'):

    if len(cluster_df) < 3:
        continue

    points = cluster_df[['lon', 'lat']].to_numpy()

    hull = ConvexHull(points)

    polygon_points = [
        [cluster_df.iloc[i]['lat'], cluster_df.iloc[i]['lon']]
        for i in hull.vertices
    ]

    folium.Polygon(
        locations=polygon_points,
        color=cluster_colors.get(cluster_id, 'gray'),
        fill=True,
        fill_color=cluster_colors.get(cluster_id, 'gray'),
        fill_opacity=0.12,
        weight=2,
        popup=f'cluster {cluster_id}'
    ).add_to(this_map)

map_data.apply(plotDot, axis=1)

this_map.fit_bounds(this_map.get_bounds())

this_map.get_root().html.add_child(Element("""
<style>
.leaflet-control-attribution {
    display: none !important;
}
</style>
"""))

this_map

In [ ]:
manager_summary = (
    selected_points_df
    .groupby('manager')
    .agg(
        Визиты=('point_id', 'size'),
        Уникальные_точки=('point_id', 'nunique'),
        Пробег_за_месяц_км=('distance_from_prev_km', 'sum'),
        Пробег_по_дорогам_км=('road_distance_from_prev_km', 'sum')
    )
)

total_summary = pd.DataFrame({
    'Визиты': [selected_points_df['point_id'].size],
    'Уникальные_точки': [selected_points_df['point_id'].nunique()],
    'Пробег_за_месяц_км': [selected_points_df['distance_from_prev_km'].sum()],
    'Пробег_по_дорогам_км': [selected_points_df['road_distance_from_prev_km'].sum()]
}, index=['Итого'])

summary = pd.concat([manager_summary, total_summary])

summary = summary.rename(index={
    0: 'Менеджер 0',
    1: 'Менеджер 1',
    2: 'Менеджер 2'
})

print(summary.round(2))

# Вывод по алгоритму 1

Классический KMeans даёт базовое географическое разделение точек, но не учитывает различную частоту посещений. Поэтому часть нагрузки распределяется неравномерно, а итоговый пробег может быть выше из-за случайного выбора точек внутри кластеров.


# Алгоритм 2: KMeans с весами и случайным выбором точек



Во втором подходе при кластеризации используется вес `n_visits`. Это означает, что точки с большим количеством требуемых посещений сильнее влияют на положение центров кластеров.

Такой подход лучше отражает реальную нагрузку, потому что часто посещаемые точки становятся важнее для географического распределения. Однако выбор точек внутри дня всё ещё остаётся случайным, поэтому баланс и маршрут могут быть не оптимальными.


In [ ]:
data = pd.read_csv('data.csv')
data.head()

data['cluster'] = KMeans(n_clusters=n_clusters, random_state=random_state).fit_predict(data[['lat', 'lon']], sample_weight=data['n_visits'])
data.groupby('cluster').size()

### Симуляция посещений для алгоритма 2

Логика месячной симуляции остаётся похожей на алгоритм 1: менеджеры ежедневно получают точки из своих кластеров с ограничением по количеству визитов в день.

Главное отличие — сами кластеры уже построены с учётом веса точки, поэтому распределение лучше связано с требуемым количеством посещений.


In [ ]:
#day1
day = 1
data_original = data.copy()

data['row_id'] = data.index
data['visits_remain'] = data['n_visits']
selected_points_df = pd.DataFrame()  # пустой, будет накапливать историю визитов

shuffled = data.sample(frac=1, random_state=42).reset_index(drop=True)

selected_points = {}
for n in range(n_managers):
    cluster_data = shuffled[shuffled['cluster'] == n]
    df_m = cluster_data.iloc[:max_visits_per_day].copy()
    df_m['manager'] = n
    df_m['day'] = day
    df_m['order_in_day'] = range(1, len(df_m) + 1)
    selected_points[n] = df_m.reset_index(drop=True)


new_batch = pd.concat(selected_points.values(), ignore_index=True)


new_batch = new_batch.sort_values(
    ['manager', 'day', 'order_in_day']
).reset_index(drop=True)

new_batch['prev_lat'] = (
    new_batch
    .groupby(['manager', 'day'])['lat']
    .shift()
)

new_batch['prev_lon'] = (
    new_batch
    .groupby(['manager', 'day'])['lon']
    .shift()
)

new_batch['distance_from_prev_km'] = new_batch.apply(
    lambda row: 0 if pd.isna(row['prev_lat']) else haversine_km(
        row['prev_lat'],
        row['prev_lon'],
        row['lat'],
        row['lon']
    ),
    axis=1
)

new_batch['road_distance_from_prev_km'] = new_batch.apply(
    lambda row: 0 if pd.isna(row['prev_lat']) else road_distance_osrm_km(
        row['prev_lat'],
        row['prev_lon'],
        row['lat'],
        row['lon']
    ),
    axis=1
)

daily_distance_road = (
    new_batch
    .groupby(['manager', 'day'])['road_distance_from_prev_km']
    .sum()
    .reset_index(name='daily_distance_road_km')
)

daily_distance_map = (
    new_batch
    .groupby(['manager', 'day'])['distance_from_prev_km']
    .sum()
    .reset_index(name='daily_distance_map_km')
)

new_batch = new_batch.merge(
    daily_distance_road, 
    on=['manager', 'day'],
    how='left'
)

new_batch = new_batch.merge(
    daily_distance_map, 
    on=['manager', 'day'],
    how='left'
)


if 'selected_points_df' in globals() and len(selected_points_df) > 0:
    selected_points_df = pd.concat([selected_points_df, new_batch], ignore_index=True)
else:
    selected_points_df = new_batch


visited_idx = new_batch['point_id']  # или другой id, который ссылается на data
data.loc[data['point_id'].isin(visited_idx), 'visits_remain'] -= 1

# удаляем полностью отработанные точки
data = data[data['visits_remain'] > 0].reset_index(drop=True)

In [ ]:


for day in range(2, n_working_days + 1):
    
    shuffled = data.sample(frac=1, random_state=42).reset_index(drop=True)

    selected_points = {}
    for n in range(n_managers):
        cluster_data = shuffled[shuffled['cluster'] == n]
        df_m = cluster_data.iloc[:max_visits_per_day].copy()
        df_m['manager'] = n
        df_m['day'] = day
        df_m['order_in_day'] = range(1, len(df_m) + 1)
        selected_points[n] = df_m.reset_index(drop=True)


    new_batch = pd.concat(selected_points.values(), ignore_index=True)


    new_batch = new_batch.sort_values(
        ['manager', 'day', 'order_in_day']
    ).reset_index(drop=True)

    new_batch['prev_lat'] = (
        new_batch
        .groupby(['manager', 'day'])['lat']
        .shift()
    )

    new_batch['prev_lon'] = (
        new_batch
        .groupby(['manager', 'day'])['lon']
        .shift()
    )

    new_batch['distance_from_prev_km'] = new_batch.apply(
        lambda row: 0 if pd.isna(row['prev_lat']) else haversine_km(
            row['prev_lat'],
            row['prev_lon'],
            row['lat'],
            row['lon']
        ),
        axis=1
    )

    new_batch['road_distance_from_prev_km'] = new_batch.apply(
    lambda row: 0 if pd.isna(row['prev_lat']) else road_distance_osrm_km(
        row['prev_lat'],
        row['prev_lon'],
        row['lat'],
        row['lon']
    ),
    axis=1
)

    daily_distance_road = (
        new_batch
        .groupby(['manager', 'day'])['road_distance_from_prev_km']
        .sum()
        .reset_index(name='daily_distance_road_km')
    )

    daily_distance_map = (
        new_batch
        .groupby(['manager', 'day'])['distance_from_prev_km']
        .sum()
        .reset_index(name='daily_distance_map_km')
    )

    new_batch = new_batch.merge(
        daily_distance_road, 
        on=['manager', 'day'],
        how='left'
    )

    new_batch = new_batch.merge(
        daily_distance_map, 
        on=['manager', 'day'],
        how='left'
    )


    if 'selected_points_df' in globals() and len(selected_points_df) > 0:
        selected_points_df = pd.concat([selected_points_df, new_batch], ignore_index=True)
    else:
        selected_points_df = new_batch


    visited_idx = new_batch['point_id']  # или другой id, который ссылается на data
    data.loc[data['point_id'].isin(visited_idx), 'visits_remain'] -= 1

    # удаляем полностью отработанные точки
    data = data[data['visits_remain'] > 0].reset_index(drop=True)

In [ ]:
from scipy.spatial import ConvexHull
import numpy as np
import folium
from folium import Element

actual = (
    selected_points_df
    .groupby('point_id')
    .size()
    .rename('visits_done')
)

map_data = (
    data_original
    .copy()
    .merge(actual, on='point_id', how='left')
)

map_data['visits_done'] = map_data['visits_done'].fillna(0).astype(int)
map_data['visits_remain'] = map_data['n_visits'] - map_data['visits_done']

map_data['status'] = np.select(
    [
        map_data['visits_done'] == 0,
        map_data['visits_remain'] <= 0
    ],
    [
        'not_started',
        'done'
    ],
    default='partial'
)

point_colors = {
    'done': 'green',
    'partial': 'orange',
    'not_started': 'red'
}

cluster_colors = {
    0: 'blue',
    1: 'purple',
    2: 'darkred'
}

this_map = folium.Map(
    prefer_canvas=True,
    attr=' '
)

def plotDot(point):
    color = point_colors[point.status]

    folium.CircleMarker(
        location=[point.lat, point.lon],
        radius=4,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        weight=2,
        popup=(
            f"point_id: {point.point_id}<br>"
            f"status: {point.status}<br>"
            f"n_visits: {point.n_visits}<br>"
            f"visits_done: {point.visits_done}<br>"
            f"visits_remain: {point.visits_remain}<br>"
            f"cluster: {point.cluster}"
        )
    ).add_to(this_map)


for cluster_id, cluster_df in map_data.groupby('cluster'):

    if len(cluster_df) < 3:
        continue

    points = cluster_df[['lon', 'lat']].to_numpy()

    hull = ConvexHull(points)

    polygon_points = [
        [cluster_df.iloc[i]['lat'], cluster_df.iloc[i]['lon']]
        for i in hull.vertices
    ]

    folium.Polygon(
        locations=polygon_points,
        color=cluster_colors.get(cluster_id, 'gray'),
        fill=True,
        fill_color=cluster_colors.get(cluster_id, 'gray'),
        fill_opacity=0.12,
        weight=2,
        popup=f'cluster {cluster_id}'
    ).add_to(this_map)

map_data.apply(plotDot, axis=1)

this_map.fit_bounds(this_map.get_bounds())

this_map.get_root().html.add_child(Element("""
<style>
.leaflet-control-attribution {
    display: none !important;
}
</style>
"""))

this_map

In [ ]:
manager_summary = (
    selected_points_df
    .groupby('manager')
    .agg(
        Визиты=('point_id', 'size'),
        Уникальные_точки=('point_id', 'nunique'),
        Пробег_за_месяц_км=('distance_from_prev_km', 'sum'),
        Пробег_по_дорогам_км=('road_distance_from_prev_km', 'sum')
    )
)

total_summary = pd.DataFrame({
    'Визиты': [selected_points_df['point_id'].size],
    'Уникальные_точки': [selected_points_df['point_id'].nunique()],
    'Пробег_за_месяц_км': [selected_points_df['distance_from_prev_km'].sum()],
    'Пробег_по_дорогам_км': [selected_points_df['road_distance_from_prev_km'].sum()]
}, index=['Итого'])

summary = pd.concat([manager_summary, total_summary])

summary = summary.rename(index={
    0: 'Менеджер 0',
    1: 'Менеджер 1',
    2: 'Менеджер 2'
})

print(summary.round(2))

# Вывод по алгоритму 2

Использование весов улучшает учёт требуемого количества посещений, но случайный выбор точек внутри дня всё ещё ограничивает качество маршрута. Алгоритм лучше отражает нагрузку, однако ему не хватает явной балансировки и оптимизации порядка посещений.


# Алгоритм 3: KMeans + веса + балансировка

Финальный вариант объединяет географическую кластеризацию, веса по количеству посещений и балансировку нагрузки. Этот подход выбран как основной для практического использования.


Подход состоит из двух этапов.

Сначала строится KMeans с весами `n_visits`, чтобы географические центры учитывали частоту посещений. Затем выполняется балансировка: точки сортируются по важности и дальности от ближайшего центра, после чего назначаются в ближайший кластер, если это не перегружает менеджера.

Идея балансировки — не просто разделить карту на зоны, а сделать так, чтобы суммарное количество требуемых визитов было распределено между менеджерами более равномерно.


In [ ]:

data_original = pd.read_csv('data.csv').reset_index(drop=True)

n_managers = 3
target_load = n_working_days * max_visits_per_day * n_managers

kmeans = KMeans(
    n_clusters=n_managers,
    random_state=42,
    n_init=10
)

kmeans.fit_predict(
    data_original[['lat', 'lon']],
    sample_weight=data_original['n_visits']
)

centers = kmeans.cluster_centers_

distances = pairwise_distances(
    data_original[['lat', 'lon']],
    centers
) #матрица расстояний от точки до центра

data_balanced = data_original.copy()
data_balanced['nearest_cluster'] = distances.argmin(axis=1)
data_balanced['nearest_distance'] = distances.min(axis=1)

manager_load = {i: 0 for i in range(n_managers)}

manager_points = {i: 0 for i in range(n_managers)}

assigned_clusters = []

# сначала распределяем тяжелые точки (самая дальняя точка от центра кластера с наибольшим количеством посещений)
order = data_balanced.sort_values(
    ['n_visits', 'nearest_distance'],
    ascending=[False, False]
).index

for idx in order:
    row = data_balanced.loc[idx]
    visits = row['n_visits']

    point_distances = pd.Series(
    distances[idx],
    index=range(n_managers),
    name='distance'
    )

    candidates = (
        point_distances
        .sort_values()
        .index
        .to_list()
    )

    chosen_cluster = None

    for cluster_id in candidates:
        if manager_load[cluster_id] + visits <= target_load:
            chosen_cluster = cluster_id
            break

    if chosen_cluster is None:
        chosen_cluster = min(
            manager_load.keys(),
            key=lambda cluster_id: manager_load[cluster_id]
        )

    assigned_clusters.append((idx, chosen_cluster))
    manager_load[chosen_cluster] += visits
    manager_points[chosen_cluster] += 1

for idx, cluster_id in assigned_clusters:
    data_balanced.loc[idx, 'cluster'] = cluster_id

data_balanced['cluster'] = data_balanced['cluster'].astype(int)

data_balanced.groupby('cluster').agg(
    points=('point_id', 'nunique'),
    required_visits=('n_visits', 'sum'),
    avg_visits=('n_visits', 'mean')
)

In [ ]:
data_balanced.sort_values(
    ['n_visits', 'nearest_distance'],
    ascending=[False, False]
).index


### Оптимизация порядка внутри дневного маршрута

После выбора точек на день их нужно упорядочить. Для этого используется жадный алгоритм ближайшего соседа: начиная с первой точки, следующей выбирается ближайшая оставшаяся точка.

Это не гарантирует математически идеальный маршрут, но даёт понятное и быстрое улучшение по сравнению со случайным порядком посещений.


In [ ]:
route_cache = {}

def order_route_nearest_neighbor(group):
    group = group.copy().reset_index(drop=True)

    if len(group) <= 1:
        group['route_order'] = range(1, len(group) + 1)
        group['distance_from_prev_km'] = 0
        return group

    remaining = group.index.tolist()
    route = []

    current_idx = remaining.pop(0)
    route.append(current_idx)

    while remaining:
        current = group.loc[current_idx]

        next_idx = min(
            remaining,
            key=lambda idx: haversine_km(
                current['lat'],
                current['lon'],
                group.loc[idx, 'lat'],
                group.loc[idx, 'lon']
            )
        )

        remaining.remove(next_idx)
        route.append(next_idx)
        current_idx = next_idx

    ordered = group.loc[route].copy().reset_index(drop=True)
    ordered['route_order'] = range(1, len(ordered) + 1)

    ordered['prev_lat'] = ordered['lat'].shift()
    ordered['prev_lon'] = ordered['lon'].shift()

    ordered['distance_from_prev_km'] = ordered.apply(
        lambda row: 0 if pd.isna(row['prev_lat']) else haversine_km(
            row['prev_lat'],
            row['prev_lon'],
            row['lat'],
            row['lon']
        ),
        axis=1
    )

    ordered['distance_from_prev_km_road'] = ordered.apply(
    lambda row: 0 if pd.isna(row['prev_lat']) else road_distance_osrm_km(
        row['prev_lat'],
        row['prev_lon'],
        row['lat'],
        row['lon']
    ),
    axis=1
)

    return ordered

### Месячное планирование по алгоритму 3

Для каждого дня и каждого менеджера выбираются доступные точки. Приоритет выбора следующий:

1. точки из собственного кластера менеджера;
2. точки, которые ещё не посещались;
3. точки с большим остатком необходимых визитов;
4. ограничение по максимальному числу визитов в день.

После выбора точки упорядочиваются внутри маршрута, а остаток визитов обновляется. Так постепенно формируется полный календарь посещений.


In [ ]:
data_work = data_balanced.copy().reset_index(drop=True)

data_work['row_id'] = data_work.index
data_work['visits_done'] = 0
data_work['visits_remain'] = data_work['n_visits']

selected_batches = []

for day in range(1, n_working_days + 1):

    used_today = set()

    for manager_id in range(n_managers):

        available = data_work[
            (data_work['visits_remain'] > 0) &
            (~data_work['point_id'].isin(used_today))
        ].copy()

        if len(available) == 0:
            continue

        # приоритет:
        # 1. Предпочесть свой cluster
        # 2. Предпочесть точки, где ещё не было визита
        # 3. Предпочесть точки с большим visits_remain
        # 4. Взять первые max_visits_per_day
        # 5. После этого переставить их в более удобном порядке
        available['is_own_cluster'] = (
            available['cluster'] == manager_id
        ).astype(int)

        available['is_new_point'] = (
            available['visits_done'] == 0
        ).astype(int)

        candidates = available.sort_values(
            ['is_own_cluster', 'is_new_point', 'visits_remain'],
            ascending=[False, False, False]
        ).head(max_visits_per_day).copy()

        candidates['manager'] = manager_id
        candidates['day'] = day

        # оптимизируем порядок точек внутри маршрута
        candidates = order_route_nearest_neighbor(candidates)

        selected_batches.append(candidates)

        used_today.update(candidates['point_id'])

        visited_rows = candidates['row_id']

        data_work.loc[
            data_work['row_id'].isin(visited_rows),
            'visits_done'
        ] += 1

        data_work.loc[
            data_work['row_id'].isin(visited_rows),
            'visits_remain'
        ] -= 1

selected_points_df = pd.concat(selected_batches, ignore_index=True)

visit_history = (
    selected_points_df
    .sort_values(['point_id', 'day'])
    .groupby('point_id')
    .agg(
        visit_days=('day', lambda x: ', '.join(map(str, sorted(x.unique())))),
        visit_count=('day', 'size'),
        managers=('manager', lambda x: ', '.join(map(str, sorted(x.unique()))))
    )
    .reset_index()
)

### Сводные метрики алгоритма 3

На этом шаге считаются итоговые показатели по менеджерам:

- общее количество выполненных визитов;
- количество уникальных посещённых точек;
- суммарный пробег маршрутов;
- пробег по дорогам, если доступен расчёт через OSRM.

Эти метрики используются для сравнения качества распределения и оценки нагрузки менеджеров.


In [ ]:
summary = (
    selected_points_df
    .groupby('manager')
    .agg(
        Визиты=('point_id', 'size'),
        Уникальные_точки=('point_id', 'nunique'),
        Пробег_км=('distance_from_prev_km', 'sum'),
        Пробег_км_дорога=('distance_from_prev_km_road', 'sum')
    )
)

total = pd.DataFrame({
    'Визиты': [selected_points_df['point_id'].size],
    'Уникальные_точки': [selected_points_df['point_id'].nunique()],
    'Пробег_км': [selected_points_df['distance_from_prev_km'].sum()],
    'Пробег_км_дорога': [selected_points_df['distance_from_prev_km_road'].sum()]
}, index=['Итого'])

summary = pd.concat([summary, total])

print(summary.round(2))

# Итоговый вывод

Алгоритм 3 показывает наиболее практичный результат среди рассмотренных подходов. Он учитывает географическую близость точек, разное количество необходимых посещений и стремится равномерно распределить нагрузку между менеджерами.

Такой подход подходит для автоматизации геопланирования: на его основе можно строить backend/frontend-приложение, которое принимает входной файл, рассчитывает маршруты и показывает результат на карте.


# Визуализация итогового решения

Ниже строится итоговая карта с зонами кластеров и маршрутами менеджеров. Слои карты можно включать и отключать, чтобы отдельно анализировать зоны ответственности и дневные маршруты.


## Итоговые таблицы и карта

Финальная часть блокнота показывает детальный результат планирования: какие точки, в какой день и в каком порядке должен посетить каждый менеджер.

Карта визуализирует зоны кластеров, точки и маршруты менеджеров. Фильтры слоёв позволяют отдельно включать и отключать зоны и маршруты, что удобно для проверки результата.


In [ ]:
# Итоговая таблица с выбранными точками
selected_points_df

In [ ]:
#Визуализация кластеров и маршрутов
route_df = selected_points_df.copy()

visit_history = (
    selected_points_df
    .sort_values(['point_id', 'day'])
    .groupby('point_id')
    .agg(
        visit_days=('day', lambda x: ', '.join(map(str, sorted(x.unique())))),
        visit_count=('day', 'size'),
        managers=('manager', lambda x: ', '.join(map(str, sorted(x.unique()))))
    )
    .reset_index()
)

route_df = route_df.merge(
    visit_history,
    on='point_id',
    how='left'
)

route_df['visit_days'] = route_df['visit_days'].fillna('не посещалась')
route_df['visit_count'] = route_df['visit_count'].fillna(0).astype(int)
route_df['managers'] = route_df['managers'].fillna('-')


map_colors = [
    'blue',
    'purple',
    'darkred',
    'green',
    'orange',
    'cadetblue',
    'darkblue',
    'darkgreen',
    'red',
    'gray'
]

cluster_colors = {
    int(cluster_id): map_colors[int(cluster_id) % len(map_colors)]
    for cluster_id in sorted(data_balanced['cluster'].unique())
}

manager_colors = {
    int(manager_id): map_colors[int(manager_id) % len(map_colors)]
    for manager_id in sorted(route_df['manager'].unique())
}

this_map = folium.Map(
    prefer_canvas=True,
    attr=' '
)

# 1. Зоны кластеров
for cluster_id, cluster_df in data_balanced.groupby('cluster'):

    cluster_layer = folium.FeatureGroup(
        name=f'Зона кластера {cluster_id}',
        show=True
    )

    cluster_points = (
        cluster_df[['lat', 'lon']]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    if len(cluster_points) >= 3:
        points = cluster_points[['lon', 'lat']].to_numpy()

        hull = ConvexHull(points, qhull_options='QJ')

        polygon_points = [
            [
                cluster_points.iloc[i]['lat'],
                cluster_points.iloc[i]['lon']
            ]
            for i in hull.vertices
        ]

        color = cluster_colors.get(cluster_id, 'gray')

        folium.Polygon(
            locations=polygon_points,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.25,
            weight=4,
            tooltip=f'Зона кластера {cluster_id}',
            popup=f'cluster {cluster_id}'
        ).add_to(cluster_layer)

    cluster_layer.add_to(this_map)


# 2. Точки и маршруты менеджеров
for manager_id, manager_df in route_df.groupby('manager'):

    color = manager_colors.get(manager_id, 'gray')

    manager_layer = folium.FeatureGroup(
        name=f'Менеджер {manager_id}',
        show=True
    )

    for _, point in manager_df.iterrows():
        folium.CircleMarker(
            location=[point['lat'], point['lon']],
            radius=4,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.85,
            weight=2,
            tooltip=folium.Tooltip(
                f"""
                <b>{point['point_id']}</b><br>
                Дни посещения: {point['visit_days']}<br>
                Кол-во визитов: {point['visit_count']}<br>
                Менеджеры: {point['managers']}<br>
                Текущий день: {point['day']}<br>
                Порядок: {point['route_order']}
                """,
                sticky=True
            ),
            popup=(
                f"point_id: {point['point_id']}<br>"
                f"manager: {point['manager']}<br>"
                f"day: {point['day']}<br>"
                f"route_order: {point['route_order']}<br>"
                f"n_visits: {point['n_visits']}<br>"
                f"visit_days: {point['visit_days']}<br>"
                f"visit_count: {point['visit_count']}<br>"
                f"managers: {point['managers']}<br>"
                f"cluster: {point['cluster']}"
            )
        ).add_to(manager_layer)

    for day, day_df in manager_df.groupby('day'):

        day_df = day_df.sort_values('route_order')

        road_route = get_osrm_route_geometry(day_df)

        if len(road_route) < 2:
            continue

        folium.PolyLine(
            locations=road_route,
            color=color,
            weight=3,
            opacity=0.65,
            tooltip=f'Менеджер {manager_id}, день {day}'
        ).add_to(manager_layer)

    manager_layer.add_to(this_map)


this_map.fit_bounds(this_map.get_bounds())

folium.LayerControl(collapsed=False).add_to(this_map)

this_map.get_root().html.add_child(Element("""
<style>
.leaflet-control-attribution {
    display: none !important;
}
</style>
"""))

this_map